# Kredi Risk Tahmini (Loan Prediction)

Başvuru sahiplerinin finansal geçmişine dayanarak bir kredinin **tamamen ödenip ödenmeyeceğini (Fully Paid)** ya da **temerrüde düşüp düşmeyeceğini (Charged Off)** öngören bir sınıflandırma modeli.

Not defterinin ilk bölümü (veri keşfi, temizlik, eksik değer doldurma) orijinal çalışmamdır; ikinci bölümde (kategorik kodlama, model eğitimi ve değerlendirme) projeyi tamamlamak için ek adımlar eklendi.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sns.set_style("darkgrid")


In [ ]:
df = pd.read_csv('LoansTrainingSet.csv')
df

In [ ]:
df. info()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df['Loan Status'].value_counts()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isna().sum()

In [ ]:
fig,ax=plt.subplots()
sns.barplot(data=df,x='Loan Status',y='Annual Income',hue='Months since last delinquent')
fig.set_size_inches([12,6])
plt.show()

In [ ]:
df['Months since last delinquent'].value_counts()

In [ ]:
fig,ax=plt.subplots()
sns.scatterplot(x='Credit Score',y='Current Loan Amount',data=df,hue='Loan Status')
fig.set_size_inches([12,6])
plt.show()

In [ ]:
df = df.drop('Years of Credit History' , axis = 1)

In [ ]:
df

In [ ]:
df.isna().sum()

In [ ]:
def missing_values(df):
        mis_val = df.isnull().sum()
        mis_val_percent = 100 * df.isnull().sum() / len(df)
        mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
        mis_val_table_ren_columns = mis_val_table.rename(
        columns = {0 : 'Missing Values', 1 : '% of Total Values'})
        mis_val_table_ren_columns = mis_val_table_ren_columns[
            mis_val_table_ren_columns.iloc[:,1] != 0].sort_values(
        '% of Total Values', ascending=False).round(1)
        print ("Dataframe has " + str(df.shape[1]) + " columns.\n"      
            "There are " + str(mis_val_table_ren_columns.shape[0]) +
              " columns that have missing values.")
        return mis_val_table_ren_columns

### Düzeltme: 'Years in current job' ve 'Maximum Open Credit'

Bir alttaki orijinal hücre `Years in current job` sütununu da sayısala çevirmeye çalışıyor, ancak değerler `"10+ years"`, `"2 years"` gibi metin olduğu için `pd.to_numeric` tüm sütunu `NaN` yapıyor ve sonraki `SimpleImputer` adımı bu yüzden hata veriyordu. Bu hücre sütunu sözlük eşlemesiyle önceden doğru şekilde sayısallaştırıyor ve `Maximum Open Credit` sütunundaki `#VALUE!` hatalı kaydı temizliyor.

In [ ]:
yil_esleme = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10,
}
df['Years in current job'] = df['Years in current job'].map(yil_esleme)
df['Maximum Open Credit'] = pd.to_numeric(df['Maximum Open Credit'], errors='coerce')

In [ ]:
# Belirli sütunları sayısal hale getir
donusturulebilir_sutunlar = ['Months since last delinquent', 'Credit Score', 'Annual Income', 'Years in current job', 'Bankruptcies', 'Tax Liens']
for col in donusturulebilir_sutunlar:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
alinacak=['Months since last delinquent', 'Credit Score', 'Annual Income', 'Years in current job', 'Bankruptcies', 'Tax Liens']
df[alinacak].head()

In [ ]:
miss_values = missing_values(df)
miss_values.head(10)

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer()
df[['Months since last delinquent','Credit Score','Annual Income', 'Years in current job','Bankruptcies','Tax Liens']] = imputer.fit_transform(df[['Months since last delinquent','Credit Score','Annual Income', 'Years in current job','Bankruptcies','Tax Liens']])


imputer = SimpleImputer(strategy='most_frequent')
df[['Years in current job' ,'Maximum Open Credit']] = imputer.fit_transform(df[['Years in current job' ,'Maximum Open Credit']])

## Kalan Temizlik ve Kategorik Değişkenlerin Kodlanması

In [ ]:
# Para birimi formatındaki sütunu sayısala çevir ($, virgül temizliği)
df['Monthly Debt'] = df['Monthly Debt'].replace(r'[\$,]', '', regex=True).astype(float)

# Hedef değişkeni ikili (binary) hale getir: Fully Paid=1, Charged Off=0
df['Loan Status'] = df['Loan Status'].map({'Fully Paid': 1, 'Charged Off': 0})

# Kalan eksik değerleri (varsa) düş
df = df.dropna()

# Kategorik sütunları one-hot encoding ile sayısala çevir
kategorik_sutunlar = ['Term', 'Home Ownership', 'Purpose']
df = pd.get_dummies(df, columns=kategorik_sutunlar, drop_first=True)

# Model için kullanılmayacak kimlik sütunlarını çıkar
df = df.drop(columns=['Loan ID', 'Customer ID'])

df.shape

## Eğitim / Test Ayrımı ve Model Eğitimi

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df.drop(columns=['Loan Status'])
y = df['Loan Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Doğruluk (Accuracy):", round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Charged Off', 'Fully Paid']))

## Karışıklık Matrisi (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Charged Off', 'Fully Paid'],
            yticklabels=['Charged Off', 'Fully Paid'])
plt.xlabel('Tahmin')
plt.ylabel('Gerçek')
plt.title('Karışıklık Matrisi')
plt.show()

## En Önemli 15 Özellik (Feature Importance)

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, ax=ax)
ax.set_title('Kredi Durumu Tahmininde En Etkili 15 Özellik')
ax.set_xlabel('Önem Skoru')
plt.tight_layout()
plt.show()